# W7C1 Lab: One Word, Many Meanings

Run every cell from the top. **Everything already works.**

The first cell downloads a small model (about 90 MB) the first time you
run it. After that it is cached and instant.

Today you will:

1. Show that a static word vector cannot tell two meanings apart.
2. Get a CONTEXTUAL vector from a real pretrained model and watch it split.
3. Use those vectors to find which sentences mean the same thing.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup.
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel

MODEL = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL)
model.eval()

print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

SENTENCES = [
    "i sat on the river bank and watched the water",
    "the fisherman slept on the muddy bank of the stream",
    "i deposited money at the bank on friday",
    "the bank approved my loan application yesterday",
]
for i, s in enumerate(SENTENCES):
    print(f"  [{i}] {s}")

## Part 1. The problem with one vector per word

word2vec gives every word exactly one vector. So 'bank' in a river sentence
and 'bank' in a money sentence get the identical vector: similarity 1.000,
no matter what the sentence says.

In [ ]:
# GIVEN. A static lookup table, the word2vec idea in three lines.
vocabulary = sorted({w for s in SENTENCES for w in s.split()})
static_table = {w: np.random.RandomState(0).randn(16) for w in vocabulary}

a = static_table["bank"]
b = static_table["bank"]
print("static vector for 'bank' in sentence 0 and sentence 2:")
print("   cosine similarity:", round(float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b))), 3))
print()
print("Exactly 1.000, and it always will be. A lookup table has no idea")
print("which sentence you are in.")

## Part 2. A contextual vector

A Transformer reads the whole sentence before it produces a vector for any
word. Same word, different sentence, different numbers.

In [ ]:
# GIVEN. Pull out the vector for one word, in context.
def vector_for(word, sentence):
    """The model's vector for `word` as it appears in `sentence`."""
    ids = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        hidden = model(**ids).last_hidden_state[0]
    tokens = tokenizer.convert_ids_to_tokens(ids["input_ids"][0])
    return hidden[tokens.index(word)]

vectors = [vector_for("bank", s) for s in SENTENCES]

sim = np.zeros((4, 4))
for i in range(4):
    for j in range(4):
        sim[i, j] = F.cosine_similarity(vectors[i], vectors[j], dim=0)

print("similarity between the four 'bank' vectors:")
print(pd.DataFrame(sim.round(3),
                   index=["river 0", "river 1", "money 2", "money 3"],
                   columns=["river 0", "river 1", "money 2", "money 3"]).to_string())

In [ ]:
# GIVEN. The same numbers as a picture.
fig, ax = plt.subplots(figsize=(4.4, 3.8))
im = ax.imshow(sim, cmap="Reds", vmin=0, vmax=1)
names = ["river 0", "river 1", "money 2", "money 3"]
ax.set_xticks(range(4), names, rotation=45); ax.set_yticks(range(4), names)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.title("'bank' against 'bank'"); plt.colorbar(im, fraction=0.046)
plt.tight_layout(); plt.show()
print("The two river sentences agree with each other. So do the two money ones.")
print("Across the two meanings, similarity drops. The model split the word.")

In [ ]:
# ================== YOUR TURN 1 ==================
# Pick another word with two meanings and test it the same way.
#
# Good candidates: bat (animal / cricket), light (not heavy / lamp),
# spring (season / coil), rock (stone / music).
#
# Write two sentences for each meaning. Use the word in lower case and
# keep it as a whole word, or the tokenizer may split it.
#
# Expected: with 'spring' the same-meaning pairs average about 0.89 and the
#           cross-meaning pairs about 0.73, so the word is clearly split. Not every
#           word works: 'bat' actually FAILS on this model, scoring 0.833 within a
#           meaning and 0.839 across. Try it and see.
# ===============================================
WORD = "spring"
SENTS = [
    "the flowers bloom in spring after the snow melts",   # season
    "spring is my favourite season of the year",          # season
    "the metal spring compressed under the heavy weight", # coil
    "he replaced a broken spring in the machine",         # coil
]

vs = [vector_for(WORD, s) for s in SENTS]
m = np.zeros((4, 4))
for i in range(4):
    for j in range(4):
        m[i, j] = F.cosine_similarity(vs[i], vs[j], dim=0)

print(pd.DataFrame(m.round(3)).to_string())
print()
same = (m[0, 1] + m[2, 3]) / 2
across = m[0, 2:].mean()
print(f"average similarity, SAME meaning:   {same:.3f}")
print(f"average similarity, ACROSS meanings: {across:.3f}")
print("split the word:", same > across)

## Part 3. Sentence vectors, and what they are for

Average all the word vectors in a sentence and you get a vector for the
whole sentence. That is how semantic search works.

In [ ]:
# GIVEN. Sentence vectors, then find the closest pair.
def sentence_vector(sentence):
    ids = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        hidden = model(**ids).last_hidden_state[0]
    return hidden.mean(dim=0)                 # average over the words

POOL = [
    "the cat sat on the mat",
    "a feline rested upon the rug",
    "the stock market closed higher today",
    "shares rose sharply this afternoon",
]
vecs = [sentence_vector(s) for s in POOL]

print("closest pairs by meaning:")
for i in range(len(POOL)):
    for j in range(i + 1, len(POOL)):
        c = F.cosine_similarity(vecs[i], vecs[j], dim=0)
        print(f"   {c:.3f}   {POOL[i][:32]:<34} | {POOL[j][:32]}")

In [ ]:
# ================== YOUR TURN 2 ==================
# Add your own pair of sentences that mean the same thing in totally
# different words, then check the model puts them together.
#
# This is exactly the query that scored 0.000 in the Week 4 search lab.
#
# Expected: paraphrases score high even with no shared words. That is the whole
#           difference between lexical search (Week 4) and semantic search: this
#           model matches meaning, not strings.
# ===============================================
MINE = [
    "the dog chased the ball across the garden",
    "a puppy ran after a toy in the yard",
]

v1, v2 = sentence_vector(MINE[0]), sentence_vector(MINE[1])
shared = set(MINE[0].split()) & set(MINE[1].split())
print(f"words in common: {shared or 'none'}")
print(f"similarity:      {F.cosine_similarity(v1, v2, dim=0):.3f}")

## Answers

Try each task before reading.

In [ ]:
# YOUR TURN 1
#   Measured gaps on this model: spring +0.165, rock +0.164, bark +0.154,
#   light +0.126, and bat -0.007, which does NOT separate. The vectors carry
#   grammar and position as well as meaning, so the sense signal is real but
#   not overwhelming. Reporting the one that fails is the honest version.
#
# YOUR TURN 2
#   Paraphrases with no shared words still score high, often above 0.6, where
#   the Week 4 tf-idf engine gave exactly 0.000 for the same kind of pair. That
#   single fact is why every modern search system embeds text instead of
#   matching words.